# LLM-as-judge Hallucination Detector (Qwen3-235B via OpenRouter)

Standalone notebook for the LLM-as-judge detector used in section 3.7 of `final_notebook.ipynb`. The detector reads a (user query, tool output, available tools, assistant response) tuple and returns character-level hallucination spans. It is prompt-only (no fine-tuning) and runs against the `dakoblov/Hallucinations` combined test split.

The system prompt encodes a three-type taxonomy (Contradiction, Overgeneration, Missing Tool) with four in-context examples (one per type plus one clean) drawn from the train split. A `TIGHT SPANS` rule with five worked examples instructs the model to return only the wrong value for value-level contradictions, which is essential to keep character precision usable on Type 1.

Predictions are returned as verbatim substrings of the answer and located back via `str.find` with a stripped-substring fallback. Inference is parallelised with 15 OpenRouter workers; the full 599-sample test runs in approximately three minutes at a cost of about \$0.90 per pass.

Requires `OPENROUTER_API_KEY` set as an environment variable, Kaggle secret, or Colab userdata.

In [ ]:
!pip install -q huggingface_hub requests tqdm pandas

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["OPENROUTER_API_KEY"] = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    try:
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        pass
assert os.environ.get("OPENROUTER_API_KEY"), "Set OPENROUTER_API_KEY before running"

## Load test and train splits from Hugging Face

Test split is the target. Train split is used only to draw four in-context examples (one per type plus one clean).

In [ ]:
import json
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

HF_DATASET_REPO = "dakoblov/Hallucinations"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

for name in ["combined_train.jsonl", "combined_test.jsonl"]:
    dest = DATA_DIR / name
    if not dest.exists():
        src = hf_hub_download(repo_id=HF_DATASET_REPO, filename=name, repo_type="dataset")
        shutil.copy(src, dest)

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train_samples = read_jsonl(DATA_DIR / "combined_train.jsonl")
test_samples = read_jsonl(DATA_DIR / "combined_test.jsonl")
print(f"train={len(train_samples)}, test={len(test_samples)}")

## Prompt design

The system prompt encodes the three-type taxonomy and a `TIGHT SPANS` rule with five worked examples. The user message contains the tool output, user query, available tools, and assistant response. The assistant must return a JSON object `{"spans": [...]}` where each span is a verbatim substring of the response.

In [ ]:
import random

SYSTEM_PROMPT = """You are a hallucination detector for assistant responses in tool-augmented dialogues. You will be given:
- The tool output (a JSON object: ground truth for what the assistant should be grounded in)
- The user query
- The list of tools the assistant has available
- The assistant's response

Your task: identify every span in the assistant's response that is a HALLUCINATION. A hallucination is any of:
1. CONTRADICTION: text that contradicts a specific value in the tool output.
2. OVERGENERATION: a factual claim, statistic, observation, historical context, or recommendation NOT supported by the tool output.
3. MISSING TOOL: a proposal to perform an action that requires a capability NOT present in the available tools list.

Important rules:
- Each span you return MUST be an EXACT verbatim substring of the assistant's response.
- Do NOT paraphrase, trim, or normalize.
- Keep spans tight.
- If the response is fully grounded, return an empty list.

TIGHT SPANS FOR VALUE-LEVEL CONTRADICTIONS:
When the hallucination is a single wrong value (number, name, date, identifier, status word), return ONLY that value.
Examples:
- Answer "Score: 9850" but tool says 4000 -> return "9850", NOT "Score: 9850".
- Answer "Alert ID is LA124." but tool says LA042 -> return "LA124", NOT "Alert ID is LA124.".
- Answer "Latitude: 35.6895" but tool gives 40.7128 -> return "35.6895".
- Answer "The NAV is **1047.15**." but tool gives 982.3 -> return "1047.15".
- Answer "Released on March 8, 2020" but tool says March 15 -> return "March 8, 2020" or "March 8".

Only widen the span when the hallucination is genuinely multi-word (a fabricated sentence, a missing-tool offer, sentence-level overgeneration).

Output ONLY a JSON object on a single line:
{"spans": ["<verbatim substring 1>", ...]}"""

USER_PROMPT_TEMPLATE = """Tool output (ground truth):
{tool_output}

User query:
{user_query}

Available tools (the assistant CAN do these):
{tools}

Assistant response:
{answer}

Now return the JSON of hallucinated substrings."""


def format_tools(tools):
    if not tools:
        return "(none)"
    lines = []
    for t in tools:
        if not isinstance(t, dict):
            continue
        name = t.get("name", "?")
        desc = (t.get("description") or "")[:200]
        lines.append(f"- {name}: {desc}")
    return "\n".join(lines)


def pick_few_shot(train_samples, seed=0):
    rng = random.Random(seed)
    def fits(s):
        return len(s.get("context", "")) <= 800 and len(s.get("output", "")) <= 500
    def span_len(s):
        labels = s.get("hallucination_labels", [])
        return int(labels[0]["end"]) - int(labels[0]["start"]) if labels else 0
    by_type = {"Type1_Contradiction": [], "Type2_Overgeneration": [], "Type3_MissingTool": [], "clean": []}
    for s in train_samples:
        if not fits(s):
            continue
        if not s["hallucination_labels"]:
            by_type["clean"].append(s)
        else:
            t = s["hallucination_labels"][0].get("type", "")
            if t in by_type:
                by_type[t].append(s)
    short_t1 = [s for s in by_type["Type1_Contradiction"] if 0 < span_len(s) <= 20]
    if short_t1:
        by_type["Type1_Contradiction"] = short_t1
    picked = []
    for t in ("Type1_Contradiction", "Type2_Overgeneration", "Type3_MissingTool", "clean"):
        if by_type[t]:
            picked.append(rng.choice(by_type[t]))
    return picked


def build_messages(sample, few_shot):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in few_shot:
        msgs.append({
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                tool_output=ex["context"],
                user_query=ex["query"],
                tools=format_tools(ex.get("tools_available", [])),
                answer=ex["output"],
            ),
        })
        gold_texts = [g["text"] for g in ex.get("hallucination_labels", [])]
        msgs.append({"role": "assistant", "content": json.dumps({"spans": gold_texts}, ensure_ascii=False)})
    msgs.append({
        "role": "user",
        "content": USER_PROMPT_TEMPLATE.format(
            tool_output=sample["context"],
            user_query=sample["query"],
            tools=format_tools(sample.get("tools_available", [])),
            answer=sample["output"],
        ),
    })
    return msgs


few_shot = pick_few_shot(train_samples)
print(f"few-shot: {len(few_shot)} examples")

## OpenRouter call, span location, parallel inference

In [ ]:
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
LLM_MODEL = "qwen/qwen3-235b-a22b-2507"
N_WORKERS = 15


def call_openrouter(messages, api_key, temperature=0.0, timeout=60):
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {"model": LLM_MODEL, "messages": messages, "temperature": temperature}
    for attempt in range(5):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, json=body, timeout=timeout)
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"]
        except Exception:
            if attempt == 4:
                raise
            time.sleep(2 ** attempt + random.random())


def extract_json(text):
    s = text.find("{"); e = text.rfind("}")
    if s < 0 or e < 0 or e <= s:
        return {}
    try:
        return json.loads(text[s:e + 1])
    except Exception:
        return {}


def locate_span(answer, substring):
    if not substring:
        return None
    idx = answer.find(substring)
    if idx >= 0:
        return idx, idx + len(substring)
    s = substring.strip()
    if s and s != substring:
        idx = answer.find(s)
        if idx >= 0:
            return idx, idx + len(s)
    return None


def detect_one(sample, api_key, few_shot):
    messages = build_messages(sample, few_shot)
    raw = call_openrouter(messages, api_key=api_key)
    parsed = extract_json(raw)
    substrings = parsed.get("spans", []) or []
    answer = sample["output"]
    spans, seen = [], set()
    for sub in substrings:
        if not isinstance(sub, str):
            continue
        pos = locate_span(answer, sub)
        if pos is None or pos in seen:
            continue
        seen.add(pos)
        spans.append({"start": pos[0], "end": pos[1], "text": answer[pos[0]:pos[1]]})
    spans.sort(key=lambda d: d["start"])
    return spans


api_key = os.environ["OPENROUTER_API_KEY"]
llm_preds = [None] * len(test_samples)
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {ex.submit(detect_one, s, api_key, few_shot): i for i, s in enumerate(test_samples)}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="LLM-as-judge"):
        i = futures[fut]
        try:
            llm_preds[i] = fut.result()
        except Exception:
            llm_preds[i] = []
print(f"Done: {sum(1 for p in llm_preds if p)} non-empty predictions of {len(llm_preds)}")

## Evaluation

Span micro F1 (RAGTruth-style character intersection), span macro F1 (per-sample F1 averaged across the split, empty-vs-empty counted as 1.0), and example-level precision / recall / F1 on the combined test set and the three per-type subsets balanced with clean samples.

In [ ]:
import pandas as pd


def _char_set(spans):
    out = set()
    for s in spans:
        out.update(range(int(s["start"]), int(s["end"])))
    return out


def span_micro_f1(samples, pred_spans):
    o = p = g = 0
    for s, ps in zip(samples, pred_spans):
        gs = _char_set(s["hallucination_labels"]); pset = _char_set(ps)
        o += len(gs & pset); p += len(pset); g += len(gs)
    pr = o / p if p else 0.0; re = o / g if g else 0.0
    f1 = 2 * pr * re / (pr + re) if (pr + re) else 0.0
    return pr, re, f1


def span_macro_f1(samples, pred_spans):
    fs = []
    for s, ps in zip(samples, pred_spans):
        gs = _char_set(s["hallucination_labels"]); pset = _char_set(ps)
        o = len(gs & pset)
        if not pset and not gs:
            fs.append(1.0); continue
        pr = o / len(pset) if pset else 0.0
        re = o / len(gs) if gs else 0.0
        fs.append(2 * pr * re / (pr + re) if (pr + re) else 0.0)
    return sum(fs) / len(fs) if fs else 0.0


def example_f1(samples, pred_spans):
    tp = fp = fn = 0
    for s, ps in zip(samples, pred_spans):
        gold = bool(s["hallucination_labels"]); pred = bool(ps)
        if gold and pred: tp += 1
        elif gold: fn += 1
        elif pred: fp += 1
    pr = tp / (tp + fp) if (tp + fp) else 0.0
    re = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * pr * re / (pr + re) if (pr + re) else 0.0
    return pr, re, f1


def filter_by_type(samples, preds, type_label):
    sel_s, sel_p = [], []
    for s, p in zip(samples, preds):
        if not s["hallucination_labels"]:
            sel_s.append(s); sel_p.append(p)
        elif s["hallucination_labels"][0].get("type") == type_label:
            sel_s.append(s); sel_p.append(p)
    return sel_s, sel_p


def all_metrics(samples, preds):
    sp, sr, sf = span_micro_f1(samples, preds)
    macro = span_macro_f1(samples, preds)
    ep, er, ef = example_f1(samples, preds)
    return {"N": len(samples), "Span P": round(sp, 3), "Span R": round(sr, 3),
            "Span F1": round(sf, 3), "Span macro F1": round(macro, 3),
            "Ex P": round(ep, 3), "Ex R": round(er, 3), "Ex F1": round(ef, 3)}


TYPE_LABELS = {
    "Type 1 + clean": "Type1_Contradiction",
    "Type 2 + clean": "Type2_Overgeneration",
    "Type 3 + clean": "Type3_MissingTool",
}

rows = [{"Split": "Combined", **all_metrics(test_samples, llm_preds)}]
for split_name, type_label in TYPE_LABELS.items():
    ss, pp = filter_by_type(test_samples, llm_preds, type_label)
    rows.append({"Split": split_name, **all_metrics(ss, pp)})
pd.DataFrame(rows)

## Save predictions

Saved as `llm_detector_predictions.jsonl` (one `{id, pred_spans}` per line). This file is bundled with the HF dataset so `final_notebook.ipynb` can load it without an API key.

In [ ]:
with open(DATA_DIR / "llm_detector_predictions.jsonl", "w") as f:
    for s, pred in zip(test_samples, llm_preds):
        f.write(json.dumps({"id": s["id"], "pred_spans": pred}) + "\n")
print(f"Saved to {DATA_DIR / 'llm_detector_predictions.jsonl'}")